In [ ]:
import csv
import multiprocessing as mp
import pickle
import sys
import time

from ipywidgets import *
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib import ticker
from matplotlib.colors import LogNorm
from matplotlib.pyplot import figure
import numpy as np
import pandas as pd
import plotly.express as px
from PyAstronomy import pyasl
import scipy.integrate as integrate
from scipy.interpolate import UnivariateSpline
from scipy.optimize import curve_fit
from scipy.stats import multivariate_normal
import six
from tqdm.auto import tqdm

import taurex.log
from taurex.binning import FluxBinner, SimpleBinner
from taurex.cache import CIACache, OpacityCache
from taurex.chemistry import TaurexChemistry
from taurex.contributions import (
    AbsorptionContribution,
    CIAContribution,
    RayleighContribution,
    SimpleCloudsContribution,
)
from taurex.data.profiles.chemistry.gas.constantgas import ConstantGas
from taurex.data.spectrum.observed import ArraySpectrum
from taurex.instruments import SNRInstrument
from taurex.model import TransmissionModel
from taurex.optimizer.nestle import NestleOptimizer
from taurex.planet import Planet
from taurex.pressure import SimplePressureProfile
from taurex.stellar import BlackbodyStar, PhoenixStar
from taurex.temperature import Isothermal

taurex.log.disableLogging()

In [2]:
OpacityCache().clear_cache()
OpacityCache().set_opacity_path("/home/patcharawee/TauREx3_public/examples/parfiles/input/xsec/xsec_sampled_R15000_0.3-15")
CIACache().set_cia_path("/home/patcharawee/TauREx3_public/examples/parfiles/input/cia/HITRAN/data")

In [ ]:
class Filters:

    def __init__(self, wavelengths, transit_depths):
        self.wavelengths = wavelengths
        self.transit_depths = transit_depths
        self.trans_funcs = pyasl.TransmissionCurves()
        self._add_sloan_filter()
        self.bandwidth = {
            "Bessel u": [3000, 4200],  # Angstroms
            "Bessel b": [3600, 5600],
            "Bessel v": [4700, 7000],
            "Bessel r": [5500, 9000],
            "Bessel i": [7000, 9200],
            "Sloan u'": [2980, 4130],
            "Sloan g'": [3630, 5830],
            "Sloan r'": [5380, 7230],
            "Sloan i'": [6430, 8630],
            "Sloan z'": [7730, 11230]
        }

    def _add_sloan_filter(self):
        sloan_u = np.loadtxt('/home/patcharawee/Thesis_work_jupyter_PhD/Sloan/u.dat')
        sloan_g = np.loadtxt('/home/patcharawee/Thesis_work_jupyter_PhD/Sloan/g.dat')
        sloan_r = np.loadtxt('/home/patcharawee/Thesis_work_jupyter_PhD/Sloan/r.dat')
        sloan_i = np.loadtxt('/home/patcharawee/Thesis_work_jupyter_PhD/Sloan/i.dat')
        sloan_z = np.loadtxt('/home/patcharawee/Thesis_work_jupyter_PhD/Sloan/z.dat')

        u_sloan_wvl, u_sloan_trans_val = sloan_u[:, 0], sloan_u[:, 3]
        g_sloan_wvl, g_sloan_trans_val = sloan_g[:, 0], sloan_g[:, 3]
        r_sloan_wvl, r_sloan_trans_val = sloan_r[:, 0], sloan_r[:, 3]
        i_sloan_wvl, i_sloan_trans_val = sloan_i[:, 0], sloan_i[:, 3]
        z_sloan_wvl, z_sloan_trans_val = sloan_z[:, 0], sloan_z[:, 3]

        self.trans_funcs.addPassband("Sloan u'", u_sloan_wvl, u_sloan_trans_val, snc=True)
        self.trans_funcs.addPassband("Sloan g'", g_sloan_wvl, g_sloan_trans_val, snc=True)
        self.trans_funcs.addPassband("Sloan r'", r_sloan_wvl, r_sloan_trans_val, snc=True)
        self.trans_funcs.addPassband("Sloan i'", i_sloan_wvl, i_sloan_trans_val, snc=True)
        self.trans_funcs.addPassband("Sloan z'", z_sloan_wvl, z_sloan_trans_val, snc=True)

    def get_filter_curve(self, filter_name):
        filter_curve = self.trans_funcs.getTransCurve(filter_name)
        weight_curve = filter_curve(self.wavelengths)
        return weight_curve / sum(weight_curve)

    def get_binned_transit_depth(self, filter_name):
        weight_curve = self.get_filter_curve(filter_name)
        return np.average(self.transit_depths, weights=weight_curve)

    def get_binned_transit_depth_error(self, filter_name):
        weight_curve = self.get_filter_curve(filter_name)
        transit_depths_average = self.get_binned_transit_depth(filter_name)
        transit_depths_variance = np.average((self.transit_depths - transit_depths_average)**2, weights=weight_curve)
        return np.sqrt(transit_depths_variance)

    def get_binned_wavelength(self, filter_name):
        weight_curve = self.get_filter_curve(filter_name)
        return np.average(self.wavelengths, weights=weight_curve)

    def get_binned_wavelength_interval(self, filter_name):
        weight_curve = self.get_filter_curve(filter_name)
        spline = UnivariateSpline(self.wavelengths, weight_curve - np.max(weight_curve) / 2, s=0)
        central_wavelength = self.get_binned_wavelength(filter_name)
        r1, r2 = spline.roots()
        return [central_wavelength - r1, r2 - central_wavelength]

    def get_binned_wavelength_list(self, filter_list):
        binned_wavelength_list = []
        for filter_name in filter_list:
            binned_wavelength_list.append(self.get_binned_wavelength(filter_name))
        return binned_wavelength_list

    def get_binned_transit_depth_list(self, filter_list):
        binned_transit_depth_list = []
        for filter_name in filter_list:
            binned_transit_depth_list.append(self.get_binned_transit_depth(filter_name))
        return binned_transit_depth_list

    def get_binned_transit_depth_error_list(self, filter_list):
        binned_transit_depth_error_list = []
        for filter_name in filter_list:
            binned_transit_depth_error_list.append(self.get_binned_transit_depth_error(filter_name))
        return binned_transit_depth_error_list

    def get_binned_wavelength_interval_list(self, filter_list):
        binned_wavelength_interval_list = []
        for filter_name in filter_list:
            binned_wavelength_interval_list.append(self.get_binned_wavelength_interval(filter_name))
        return binned_wavelength_interval_list

    def get_band_flux(self, filter_list):
        band_flux = []
        for filter_name in filter_list:
            band_flux.append(integrate.simpson(self.transit_depths * self.get_filter_curve(filter_name), self.wavelengths, dx=0.2))
        return np.array(band_flux)

    def get_errorbar(self, filter_list):  # 1%
        band_flux = self.get_band_flux(filter_list)
        C = (100 / self.get_binned_transit_depth("Sloan r'"))**2 / band_flux[7]
        return 1 / np.sqrt(C * self.get_band_flux(filter_list))


############################################################

def calculate_semi_major_axis(period_days):
    G = 6.67430e-11   # (m^3 kg^-1 s^-2)
    M_sun = 1.989e30  # (kg)
    AU = 1.496e11     # 1 AU = 1.496e11 m

    P = period_days * 86400  # (s)
    a_cubed = (G * M_sun * P**2) / (4 * np.pi**2)  # a^3
    a_meters = a_cubed**(1 / 3)  # a
    a_AU = a_meters / AU  # convert to AU
    return a_AU


def find_T_from_R(num_samples=1000, R_min_jupiter=0.5, R_max_jupiter=1.4):

    EARTH_TO_JUPITER_RADIUS = 11.2

    R_min = R_min_jupiter * EARTH_TO_JUPITER_RADIUS
    R_max = R_max_jupiter * EARTH_TO_JUPITER_RADIUS

    coeff_mean, coeff_std = 1.10, 0.15
    b1_mean, b1_std = 0.35, 0.02

    T_list = []
    R_list = []
    coeff_list = []
    b1_list = []

    while len(T_list) < num_samples:
        R_value = np.random.uniform(R_min, np.nextafter(R_max, np.inf))  # for half-opened interval [Rmin, Rmax]
        coeff_value = np.random.normal(coeff_mean, coeff_std)
        b1_value = np.random.normal(b1_mean, b1_std)

        T_value = (R_value / coeff_value)**(1 / b1_value)

        if 500 <= T_value <= 4000:
            T_list.append(T_value)
            R_list.append(R_value / EARTH_TO_JUPITER_RADIUS)
            coeff_list.append(coeff_value)
            b1_list.append(b1_value)

    df = pd.DataFrame({
        'T (K)': T_list,
        'R (R_Jupiter)': R_list,
        'Coeff': coeff_list,
        'b1': b1_list
    })

    return df


N = 100
R_min = 0.8
R_max = 1.0
df_result = find_T_from_R(num_samples=N, R_min_jupiter=R_min, R_max_jupiter=R_max)
filename = 'Syn100_uniform_1.pickle'

print('START!')
save_data = []

for i in tqdm(range(N), desc='Main Loop', leave=True):

    # default
    Ts = 5500
    Rs = 1    # Solar
    Ms = 1    # Solar
    met = 1   # Solar
    dist = 1  # dist from Earth (pc)
    magK = 10

    if met <= 0:
        star = PhoenixStar(
            temperature=Ts,      # K
            radius=Rs,           # R_sun
            distance=dist,       # pc (from earth)
            magnitudeK=magK,     # mag in K band
            mass=Ms,             # M_sun
            metallicity=met,     # Solar unit
            phoenix_path="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/PHOENIX-ACES-AGSS-COND-2011/Z-%.1f" % np.abs(met),
            retro_version_file="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/WAVE_PHOENIX-ACES-AGSS-COND-2011.fits"
        )
    else:
        star = PhoenixStar(
            temperature=Ts,
            radius=Rs,
            distance=dist,
            magnitudeK=magK,
            mass=Ms,
            metallicity=met,
            phoenix_path="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/PHOENIX-ACES-AGSS-COND-2011/Z+%.1f" % met,
            retro_version_file="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/WAVE_PHOENIX-ACES-AGSS-COND-2011.fits"
        )

    pressure_profile = SimplePressureProfile(
        nlayers=100,
        atm_min_pressure=0.0001,
        atm_max_pressure=1000000.0
    )  # default

    # default
    M_min = 115 / 317.8
    M_max = 10000 / 317.8
    P = 1  # days
    a = calculate_semi_major_axis(P)
    b = 0.5
    Rp = np.random.uniform(R_min, R_max)
    Mp = 1
    Tp = np.random.uniform(500, 4000)
    coeff_value = df_result['Coeff'][i]
    t14 = 3000  # s
    A = np.random.uniform(0.1, 0.4)

    planet = Planet(
        planet_mass=Mp,
        planet_radius=Rp,
        planet_distance=a,
        impact_param=b,
        orbital_period=P,
        albedo=A,
        transit_time=t14
    )
    
    temperature_profile = Isothermal(T=Tp)
    ran_ratio = 0.1
    chemistry = TaurexChemistry(
        fill_gases=['H2', 'He'],
        ratio=ran_ratio
    )

    TiO = 10**np.random.uniform(-4, -1)
    VO = 10**np.random.uniform(-4, -1)

    chemistry.addGas(ConstantGas('TiO', mix_ratio=TiO))
    chemistry.addGas(ConstantGas('VO', mix_ratio=VO))

    tm = TransmissionModel(
        planet=planet,
        star=star,
        pressure_profile=pressure_profile,
        temperature_profile=temperature_profile,
        chemistry=chemistry,
        new_path_method=False
    )
    
    tm.add_contribution(AbsorptionContribution())
    tm.add_contribution(CIAContribution(cia_pairs=['H2-H2', 'H2-He']))
    tm.add_contribution(RayleighContribution())
    tm.build()
    tm.model()

    filter_list = {
        "Bessel u": "blue",
        "Bessel b": "teal",
        "Bessel v": "yellowgreen",
        "Bessel r": "gold",
        "Bessel i": "orangered",
        "Sloan u'": "purple",
        "Sloan g'": "green",
        "Sloan r'": "yellow",
        "Sloan i'": "darkorange",
        "Sloan z'": "red"
    }

    native_grid, rprs, tau, _ = tm.model()
    wavelengths = np.flip(10000 / native_grid) * 1e4  # micron -> A
    transit_depths = np.flip(rprs)
    
    binned_filters = Filters(wavelengths, transit_depths)
    bt = np.array(binned_filters.get_binned_transit_depth_list(filter_list))
    bw = np.array(binned_filters.get_binned_wavelength_list(filter_list))
    bw_err = np.array(binned_filters.get_binned_wavelength_interval_list(filter_list)).T
    bt_err = binned_filters.get_errorbar(filter_list)

    filters_dict = {
        'U': "blue",
        'B': "teal",
        'V': "yellowgreen",
        'R': "gold",
        'I': "orangered",
        'u': "purple",
        'g': "green",
        'r': "yellow",
        'i': "darkorange",
        'z': "red"
    }

    bin_data = {}
    for name, bw_data, depth_data, bw_low_err_data, bw_up_err_data, bt_err_data in zip(
        list(filters_dict), list(bw.T), list(bt.T), list(bw_err[0].T), list(bw_err[1].T), list(bt_err.T)
    ):
        bin_data[name + '_bw'] = bw_data
        bin_data[name] = depth_data
        bin_data[name + '_bw_err_low'] = bw_low_err_data
        bin_data[name + '_bw_err_up'] = bw_up_err_data
        bin_data[name + '_err'] = bt_err_data

    params = {
        'pl_orbper': P,
        'pl_orbsmax': a,
        'pl_radj': Rp,
        'pl_massj': Mp,
        'pl_eqt': Tp,
        'Coeff': coeff_value,
        'st_teff': Ts,
        'st_rad': Rs,
        'st_mass': Ms,
        'MR_TiO': TiO,
        'MR_VO': VO
    }
    
    res = params | bin_data
    save_data.append(res)

with open(filename, 'wb') as handle:
    pickle.dump(save_data, handle, protocol=pickle.HIGHEST_PROTOCOL)
    time.sleep(0.1)

In [ ]:
N = 10000
R_min = 0.8
R_max = 1.0
df_result = find_T_from_R(num_samples=N, R_min_jupiter=R_min, R_max_jupiter=R_max)
filename = 'Syn100_uniform_1.pickle'

print('START!')
save_data = []

for i in tqdm(range(N), desc='Main Loop', leave=True):

    # default
    Ts = 5500
    Rs = 1    # Solar
    Ms = 1    # Solar
    met = 1   # Solar
    dist = 1  # dist from Earth (pc)
    magK = 10

    if met <= 0:
        star = PhoenixStar(
            temperature=Ts,      # K
            radius=Rs,           # R_sun
            distance=dist,       # pc (from earth)
            magnitudeK=magK,     # mag in K band
            mass=Ms,             # M_sun
            metallicity=met,     # Solar unit
            phoenix_path="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/PHOENIX-ACES-AGSS-COND-2011/Z-%.1f" % np.abs(met),
            retro_version_file="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/WAVE_PHOENIX-ACES-AGSS-COND-2011.fits"
        )
    else:
        star = PhoenixStar(
            temperature=Ts,
            radius=Rs,
            distance=dist,
            magnitudeK=magK,
            mass=Ms,
            metallicity=met,
            phoenix_path="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/PHOENIX-ACES-AGSS-COND-2011/Z+%.1f" % met,
            retro_version_file="/home/patcharawee/Thesis_work_jupyter_PhD/Phoenix/phoenix.astro.physik.uni-goettingen.de/v2.0/HiResFITS/WAVE_PHOENIX-ACES-AGSS-COND-2011.fits"
        )

    pressure_profile = SimplePressureProfile(
        nlayers=100,
        atm_min_pressure=0.0001,
        atm_max_pressure=1000000.0
    )  # default

    # default
    M_min = 115 / 317.8
    M_max = 10000 / 317.8
    P = 1  # days
    a = calculate_semi_major_axis(P)
    b = 0.5
    Rp = np.random.uniform(R_min, R_max)
    Mp = 1
    Tp = np.random.uniform(500, 4000)
    coeff_value = df_result['Coeff'][i]
    t14 = 3000  # s
    A = np.random.uniform(0.1, 0.4)

    planet = Planet(
        planet_mass=Mp,
        planet_radius=Rp,
        planet_distance=a,
        impact_param=b,
        orbital_period=P,
        albedo=A,
        transit_time=t14
    )
    
    temperature_profile = Isothermal(T=Tp)
    ran_ratio = 0.1
    chemistry = TaurexChemistry(
        fill_gases=['H2', 'He'],
        ratio=ran_ratio
    )

    TiO = 10**np.random.uniform(-4, -1)
    VO = 10**np.random.uniform(-4, -1)

    chemistry.addGas(ConstantGas('TiO', mix_ratio=TiO))
    chemistry.addGas(ConstantGas('VO', mix_ratio=VO))

    tm = TransmissionModel(
        planet=planet,
        star=star,
        pressure_profile=pressure_profile,
        temperature_profile=temperature_profile,
        chemistry=chemistry,
        new_path_method=False
    )
    
    tm.add_contribution(AbsorptionContribution())
    tm.add_contribution(CIAContribution(cia_pairs=['H2-H2', 'H2-He']))
    tm.add_contribution(RayleighContribution())
    tm.build()
    tm.model()

    filter_list = {
        "Bessel u": "blue",
        "Bessel b": "teal",
        "Bessel v": "yellowgreen",
        "Bessel r": "gold",
        "Bessel i": "orangered",
        "Sloan u'": "purple",
        "Sloan g'": "green",
        "Sloan r'": "yellow",
        "Sloan i'": "darkorange",
        "Sloan z'": "red"
    }

    native_grid, rprs, tau, _ = tm.model()
    wavelengths = np.flip(10000 / native_grid) * 1e4  # micron -> A
    transit_depths = np.flip(rprs)
    
    binned_filters = Filters(wavelengths, transit_depths)
    bt = np.array(binned_filters.get_binned_transit_depth_list(filter_list))
    bw = np.array(binned_filters.get_binned_wavelength_list(filter_list))
    bw_err = np.array(binned_filters.get_binned_wavelength_interval_list(filter_list)).T
    bt_err = binned_filters.get_errorbar(filter_list)

    filters_dict = {
        'U': "blue",
        'B': "teal",
        'V': "yellowgreen",
        'R': "gold",
        'I': "orangered",
        'u': "purple",
        'g': "green",
        'r': "yellow",
        'i': "darkorange",
        'z': "red"
    }

    bin_data = {}
    for name, bw_data, depth_data, bw_low_err_data, bw_up_err_data, bt_err_data in zip(
        list(filters_dict), list(bw.T), list(bt.T), list(bw_err[0].T), list(bw_err[1].T), list(bt_err.T)
    ):
        bin_data[name + '_bw'] = bw_data
        bin_data[name] = depth_data
        bin_data[name + '_bw_err_low'] = bw_low_err_data
        bin_data[name + '_bw_err_up'] = bw_up_err_data
        bin_data[name + '_err'] = bt_err_data

    params = {
        'pl_orbper': P,
        'pl_orbsmax': a,
        'pl_radj': Rp,
        'pl_massj': Mp,
        'pl_eqt': Tp,
        'Coeff': coeff_value,
        # 'b1': b1_value,
        # 'b2': b2_value,
        'st_teff': Ts,
        'st_rad': Rs,
        'st_mass': Ms,
        'MR_TiO': TiO,
        'MR_VO': VO
    }
    
    res = params | bin_data
    save_data.append(res)

with open(filename, 'wb') as handle:
    pickle.dump(save_data, handle, protocol=pickle.HIGHEST_PROTOCOL)
    time.sleep(0.1)

START!


Main Loop:   0%|          | 0/100 [00:00<?, ?it/s]

In [13]:
with open(filename, 'wb') as handle:
    pickle.dump(save_data, handle, protocol = pickle.HIGHEST_PROTOCOL)

In [14]:
loaded_data = pd.read_pickle(filename)
df = pd.DataFrame.from_dict(loaded_data)
df

,pl_orbper,pl_orbsmax,pl_radj,pl_massj,pl_eqt,Coeff,b1,st_teff,st_rad,st_mass,...,i_bw,i,i_bw_err_low,i_bw_err_up,i_err,z_bw,z,z_bw_err_low,z_bw_err_up,z_err
0,1,0.019572,0.993796,23.236780,633.222264,1.186291,0.347066,5500,1,1,...,7486.022892,0.010441,583.726235,685.432784,9.470617e-07,8955.060521,0.010439,684.172662,494.268237,8.659493e-07
1,1,0.019572,1.163233,14.891700,1791.409323,1.059112,0.335038,5500,1,1,...,7486.022892,0.014326,583.726235,685.432784,1.299383e-06,8955.060521,0.014324,684.172662,494.268237,1.188099e-06
2,1,0.019572,1.031154,29.418334,645.994941,1.366850,0.329802,5500,1,1,...,7486.022892,0.011244,583.726235,685.432784,1.019922e-06,8955.060521,0.011243,684.172662,494.268237,9.325828e-07
3,1,0.019572,1.196026,12.653324,853.222067,1.276226,0.348348,5500,1,1,...,7486.022892,0.015150,583.726235,685.432784,1.374157e-06,8955.060521,0.015149,684.172662,494.268237,1.256475e-06
4,1,0.019572,0.912065,1.633420,530.612213,1.205996,0.340541,5500,1,1,...,7486.022892,0.008954,583.726235,685.432784,8.125933e-07,8955.060521,0.008938,684.172662,494.268237,7.436112e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1,0.019572,1.048429,19.515670,716.841915,1.116614,0.357864,5500,1,1,...,7486.022892,0.011633,583.726235,685.432784,1.055198e-06,8955.060521,0.011632,684.172662,494.268237,9.648456e-07
9996,1,0.019572,0.926161,12.900390,674.132878,1.206868,0.330268,5500,1,1,...,7486.022892,0.009072,583.726235,685.432784,8.230065e-07,8955.060521,0.009071,684.172662,494.268237,7.525450e-07
9997,1,0.019572,1.061037,18.499338,1218.701410,0.772073,0.384747,5500,1,1,...,7486.022892,0.011942,583.726235,685.432784,1.083651e-06,8955.060521,0.011937,684.172662,494.268237,9.910080e-07
9998,1,0.019572,0.954957,11.573223,1869.871958,1.051093,0.307952,5500,1,1,...,7486.022892,0.009708,583.726235,685.432784,8.804496e-07,8955.060521,0.009705,684.172662,494.268237,8.051330e-07
